# 프로그래밍 권장 사항(Programming Recommendations)

## 실무 팁!

### CPython 최적화에 의존하지 말 것 (특히 문자열 더하기)

In [ ]:
# ❌ 피해야 할 패턴 (루프 안에서)
# - CPython은 일부 경우 이걸 최적화해주지만,
# - 다른 구현(PyPy, Jython 등)에서는 성능이 O(n²) 가 될 수 있음.
s = ""
for part in parts:
    s += part      # 또는 s = s + part

In [ ]:
# ✔ 권장:
# - “루프 안에서 문자열 더하기”는 무조건 join으로 바꾼다고 생각하면 편하다.
# - 로그를 쌓을 때도 "".join 또는 f-string + 리스트에 모았다가 한 번에 출력하는 게 좋다.
s = "".join(parts)   # O(n), 구현 간 일관된 성능

### None 비교는 항상 is / is not

In [ ]:
x = True

# ✔ 권장:
print(x is None)  # None 인지 검사. None 이 아니면 False
print(x is not None)

# ❌ 나쁜 예:
print(x == None)
print(x != None)

False
True
True
False
True


### `if x` 는 **그냥 값이 있는냐/비어 있는냐**를 검사할 때 사용하라.

In [ ]:
# 파이썬에서 빈 값은 모두 False로 간주된다.

def check(x):
    if x:  # 값이 있는지 검사
        return True
    else:
        return False

# 빈 값으로 간주된다.
print(check(None))
print(check(False))
print(check(0))
print(check(0.0))
print(check(0j))

# 빈 시퀀스도 빈 값으로 간주된다.
print(check(""))
print(check([]))
print(check(()))
print(check(range(0)))

# 빈 컨테이너도 빈 값으로 간주된다.
print(check({}))
print(check(set()))
print(check({}.keys()))

print("-----------------")

# 빈 값이 아닌 경우
print(check(True))
print(check(1))
print(check(1.1))
print(check(3j))

# 시퀀스의 요소가 하나라도 있으면 빈 값이 아니다.
print(check("Hello"))
print(check([1]))
print(check((0,)))
print(check(range(1)))

# 컨테이너의 요소가 하나라도 있으면 빈 값이 아니다.
print(check({"name": "Alice"}))
print(check(set([0])))
print(check({"name": "Alice", "age": 20}.keys()))


False
False
False
False
False
False
False
False
False
False
False
False
-----------------
True
True
True
True
True
True
True
True
True
True
True


### `is not` vs `not ... is`

In [ ]:
# ✔ 권장:
if foo is not None:
    ...

# ❌ 나쁜 예: 위와 동일하지만 가독성이 떨어진다.
if not foo is None:
    ...

- 비교 연산자 구현 시 6개를 다 구현하거나 `@total_ordering` 데코레이터 사용
    - `__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__` 6개를 다 구현하는 게 가장 안전
    - 하지만 매번 6개 다 쓰기 귀찮으니 `functools.total_ordering` 데코레이터를 사용하여 편리하게 구현 가능
        - `__eq__` 구현
        - `<`, `<=`, `>`, `>=` 비교 연산자 중 하나만 구현하면 나머지 비교 연산자는 자동 생성됨

In [38]:
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, major):
        self.major = major

    # __eq__ 메서드는 반드시 구현해야 한다.
    def __eq__(self, other):
        return self.major == other.major

    # 그 외 <, <=, >, >= 에 해당하는 메서드를 한 개 이상 구현해야 한다.
    def __lt__(self, other):
        return self.major < other.major

    # 그러면 total_ordering 데코레이터가 나머지 비교 연산자들을 자동으로 생성해준다.

v1 = Version(1)
v2 = Version(2)

print(v1 < v2)    # True
print(v1 <= v2)   # True
print(v1 == v2)   # False
print(v1 != v2)   # True
print(v1 > v2)    # False
print(v1 >= v2)   # False   

True
True
False
True
False
False


### lambda를 이름에 직접 바인딩하지 말 것

In [ ]:
# ✔ 권장: 그냥 함수를 정의하라.
def f(x):
    return 2 * x

# ❌ 나쁜 예: 이렇게 하지 말 것
f = lambda x: 2 * x

# lambda는 표현식 안에서 한 번만 쓰는 아주 짧은 함수일 때 사용하라.
sorted(items, key=lambda x: x.name)  # 이런 건 OK

### 예외 설계: `Exception` 상속 + 계층 구조 + 이름

- `BaseException` 직접 상속 금지

In [ ]:
# ✔ 권장:
class MyError(Exception):
    ...

# ❌ 나쁜 예:
# - BaseException은 SystemExit, KeyboardInterrupt 같은 프로그램 흐름 중단용에 쓰는 특수 계층이라, 
#   직접 상속은 거의 안 쓴다.
class MyError(BaseException):
    ...

- “무엇이 잘못됐는지” 기준으로 계층 설계
    - “어디서 발생했는지” 기준이 아니라
    - "catch 하는 쪽에서 **어떤 종류의 문제로 구분해서 처리할지**” 기준으로 설계

In [ ]:
class AppError(Exception): ...
class ConfigError(AppError): ...
class DatabaseError(AppError): ...
class NetworkError(AppError): ...

- `Error` 접미사
    - 에러라면 이름 끝에 Error 붙이는 것을 권장.
    - `StopIteration`, `Cancelled` 같은 “제어 흐름용 예외”는 꼭 Error가 아니어도 된다.

In [ ]:
class InvalidTokenError(AppError):
    ...

### 예외 체이닝: `raise X from Y`

- 원래 예외 스택 트레이스를 보존하면서 상위 레벨에서 더 의미 있는 예외로 바꿔 던질 때 유용

In [ ]:
try:
    something()
except ValueError as e:
    raise ConfigError("invalid config") from e

In [ ]:
# 내부 예외를 감추고 싶을 때,
# - 대신 원래 메시지나 키 이름 등을 포함시키는 게 좋다.
raise ConfigError("invalid config") from None

### 예외 잡을 때: 구체적인 예외를 잡고, bare except:는 최소화

In [ ]:
# ✔ 권장:
try:
    import platform_specific_module
except ImportError:
    platform_specific_module = None

# ❌ 나쁜 예: bare exception
# - SystemExit, KeyboardInterrupt까지 다 잡아 버림
# - Ctrl+C로 종료도 잘 안 되고, 진짜 버그도 숨겨짐
try:
    ...
except:
    ...

# ✔ 권장: 모든 에러 잡고 싶다면 다음 방식이 더 낫다.
try:
    ...
except Exception:
    ...

- bare except 를 써도 괜찮은 상황

In [ ]:
# traceback을 출력/로그 남기고 끝낼 때,
try:
    ...
except:
    logging.exception("Unexpected error")
    raise

# 또는 cleanup 만 하고 다시 예외를 올릴 때: 대부분 try-finally를 쓰는 게 더 낫다.

### try/except 범위는 최소화

- 실무에서는 “딱 그 줄만” 감싸는 걸 목표로 하는 게 좋다.

In [ ]:
# ✔ 권장:
try:
    value = collection[key]
except KeyError:
    return key_not_found(key)
else:
    return handle_value(value)


# ❌ 나쁜 예: 너무 넓은 범위의 try/except
# 지양
try:
    return handle_value(collection[key])  # 너무 넓음
except KeyError:
    return key_not_found(key)            # handle_value() 내부 KeyError도 잡아버림

### 자원 관리: with 사용

- 파일, DB 연결, 락 등은 항상 with로 감싸서 사용
- context manager가 단순 “open/close”가 아니고 특별한 동작(트랜잭션 시작 등)을 한다면, 의미가 드러나는 메서드 이름으로 감싸서 노출하는 게 좋다.

In [ ]:
# ✔ 권장:
with open(path) as f:  # 블록이 끝나면 자동으로 파일을 닫는다.
    data = f.read()

with conn.begin_transaction():  # 블록이 끝나면 자동으로 커밋/롤백한다.
    do_stuff(conn)

with config.temporary_mode("debug"):  # 블록이 끝나면 디버그 모드에서 원래 모드로 복원한다.
    run_critical_logic()   

with user.elevated_privileges():  # 블록이 끝나면 원래 권한으로 복원한다.
    perform_admin_tasks()

with mutext.lock():  # 블록이 끝나면 자동으로 락을 해제한다.
    critical_section()    

# ❌ 나쁜 예: 
with conn:  # 블록이 끝났을 때 트랜잭션을 닫는 것인지 아니면 단순히 연결을 닫는 것인지 알 수 없다.
    do_stuff(conn)

with settings:  # 설정을 로딩하는 건지? 설정을 잠시 바꾸는 건지? 파일을 여는 건지? 알 수 없다.
    do_stuff()

### return 일관성

- 한 함수 안에서 모든 return이 표현식을 반환하거나,
- 전부 값 없이 return (또는 끝까지 가서 암시적 None)이어야 함.
- 섞여 있으면 읽는 사람이 “이 함수는 항상 값이 있는 건가? 아닐 수도 있는 건가?” 혼란.

In [ ]:
# ✔ 권장:
def foo(x):
    if x >= 0:
        return math.sqrt(x)
    else:
        return None  # 명시적으로 None 반환

def bar(x):
    if x < 0:
        return None  # 명시적으로 None 반환
    return math.sqrt(x)

# ❌ 나쁜 예: 
def foo(x):
    if x >= 0:
        return math.sqrt(x)
    # 끝까지 가서 암시적 None

def bar(x):
    if x < 0:
        return  # 암시적 None
    return math.sqrt(x)

### 문자열 접두/접미 검사: slice 대신 `startswith()` / `endswith()`

- startswith / endswith는 더 읽기 쉽고, 길이가 다른 경우, 빈 문자열 등 엣지 케이스에도 안전

In [ ]:
# ✔ 권장:
if foo.startswith("bar"):
    ...

# ❌ 나쁜 예: 
if foo[:3] == "bar":
    ...

### 타입 비교는 `type(...)` 말고 `isinstance()`

- 상속까지 포함해서 체크하기 때문에 duck typing이 가능함
    - 즉 서브 클래스도 허용한다.
- type()을 사용하면 duck typing을 해칠 수 있음
    - 즉 서브 클래스를 허용하지 않는다.
- duck typing?
    - 어떤 객체가 특정 메서드나 속성을 가지고 있다면, 그 객체를 특정 타입으로 간주하는 프로그래밍 스타일
    - 의미: 오리처럼 걷고, 오리처럼 꽥꽥거리는 객체는 오리로 간주

In [ ]:
# ✔ 권장:
if isinstance(obj, int):
    ...

# ❌ 나쁜 예: 
if type(obj) is int:
    ...
if type(obj) is type(1):
    ...


### 시퀀스 비어 있음 체크:  `if seq` / `if not seq`

In [ ]:
# ✔ 권장:
if not seq: ...
if seq: ...

# ❌ 나쁜 예: 
if len(seq): ...
if not len(seq): ...


### 불린값은 True/False와 == 비교하지 말 것

- “진리값” 확인하려고 True와 비교할 필요가 없다.
- if x / if not x로 충분합니다.

In [ ]:
# ✔ 권장:
if is_valid: ...
if not is_valid: ...

# ❌ 나쁜 예: 
if is_valid == True: ...
if greeting is True: ... # 더 안 좋음


### finally 블록 안에서 `return` / `break` / `continue`는 가능한 사용하지 말라.

- try 안에서 예외가 발생해도, finally의 return 때문에 예외가 먹혀버림
- 디버깅이 극도로 어려워짐
- 실무:
    - finally 안에서는 cleanup만 하고,
    - 흐름 제어는 밖에서 하도록 설계하는 게 좋다.

In [ ]:
# ❌ 나쁜 예:
def foo():
    try:
        1 / 0
    finally:
        return 42   # 예외가 조용히 사라짐


### 기타

- 문자열 리터럴에서 눈에 안 보이는 trailing whitespace에 의존하지 말 것
    - 편집기/포매터가 자동으로 지워버릴 수 있음

## 함수 어노테이션(Function Annotations)

- 파이썬이 PEP 484(Type Hints)를 채택하면서,
    - 함수 어노테이션은 오직 타입 힌트 용도로 간주된다.
    - 함수 어노테이션을 다른 목적(문서 등)으로 사용하는 것은 권장하지 않는다.
    - 타입 힌트의 문법은 PEP 484 기준에 따른다.

In [ ]:
# ✔ 권장:
def add(x: int, y: int) -> int:
    return x + y

### Annotation을 다른 목적(문서 등)에 쓰고 싶다면?

- 파일 상단에 `# type: ignore` 주석을 추가해서 타입 검사기에게 무시하라고 알린다.
    - 타입 검사기는 이 파일에서 함수 어노테이션을 타입 힌트로 간주하지 않는다.

In [ ]:
# type: ignore
def foo(x: "the answer") -> "ignored":
    ...

### 타입 체커는 Python 인터프리터와 별개

- Python 인터프리터는 annotation을 무시한다.
    - annotation은 런타임에 강제되지 않음
    - annotation이 틀려도 실행은 가능
    - annotation이 있어도 실행 속도에는 영향 없음
- 타입 체크는 옵션이며 MyPy/Pyright 같은 외부 도구가 담당한다.
- annotation은 “강제 규칙”이 아닌 정적 분석용 메타데이터로 취급된다.

### 라이브러리 사용자에게 타입 정보를 제공하려면 → Stub 파일(.pyi)

- `.pyi`(type stub) 파일은 “타입 정보만 들어 있는, 코드 없는 파일”이다.
- MyPy와 Pyright는 `.py`를 읽기 전에 `.pyi`를 먼저 읽는다.
- 라이브러리 개발자가 `.py`를 복잡하게 수정하지 않고도 타입 정보 제공할 수 있다.

In [ ]:
# 프로젝트 구조
example_project/
    foo.py
    foo.pyi
    main.py

In [ ]:
# foo.py

def add(x, y):
    return x + y

def greet(name):
    return f"Hello, {name}!"

In [ ]:
# foo.pyi
# - 파이썬 실행에는 영향이 없다.
# - 런타임에서는 .pyi 파일을 전혀 사용하지 않는다.

def add(x: int, y: int) -> int: ...
def greet(name: str) -> str: ...

In [ ]:
# main.py

from foo import add, greet

print(add(1, 2))
print(greet("Alice"))


## 변수 어노테이션(Variable Annotations)

### 기본 규칙 (콜론과 공백 규칙)

- 규칙 1 — 콜론(:) 앞에는 공백 없음
- 규칙 2 — 콜론(:) 뒤에는 공백 하나

In [ ]:
x: int
name: str
values: List[int]

### 할당이 있는 경우 (= 사용 시 공백 규칙)

-   `=` 양쪽에 공백 하나씩 넣어야 함

In [ ]:
# ✔ 권장:
label: str = "<unknown>"
count: int = 0

# ❌ 나쁜 예:
code:int        # ❌ 콜론 뒤에 공백 없음
code : int      # ❌ 콜론 앞에 공백 넣음

class Test:
    result: int=0      # ❌ = 양쪽에 공백 없음
    flag:bool =True    # ❌ 콜론 뒤 공백 없음, = 뒤 공백 없음


### 모듈 변수 / 클래스 변수 / 인스턴스 변수 / 지역 변수 모두 동일 규칙

- 모든 종류의 변수에 일관되게 적용해야 한다.

In [ ]:
# 모듈 변수
VERSION: str = "1.0"
DEBUG: bool = False

# 클래스 변수
class Point:
    coords: Tuple[int, int]
    label: str = "<unknown>"

# 인스턴스 변수 (생성자 내부)
class User:
    def __init__(self, name: str):
        self.name: str = name
        self.age: int = 0

# 지역 변수
def process() -> None:
    items: List[str] = []
    count: int = 5

### Stub 파일(.pyi)에서는 변수 annotation 문법이 기본

- 변수 annotation 문법은 stub 파일(.pyi)에서 모든 Python 버전에서 기본 문법이다.
    - `.pyi` 파일에서는 PEP 526 변수 어노테이션 문법만 사용해야 한다.

In [ ]:
# foo.pyi

# 변수
x: int
items: List[str]
db_url: str = ...

# 함수
def add(x: int, y: int) -> int: ...

# 클래스
class User:
    name: str
    age: int
    def greet(self) -> str: ...

# 모듈 변수
VERSION: str


- PEP 526 이전 방식(참고만 하라!)

In [ ]:
# 타입 주석을 사용했다.

x = 1  # type: int
items = []  # type: List[str]

def add(x, y):
    # type: (int, int) -> int
    return x + y

### 실무에서 자주 사용하는 패턴

- None 가능한 타입은 Optional 또는 | None 사용

In [ ]:
token: Optional[str] = None

# Python 3.10 이상에서 사용 가능
token: str | None = None

- 타입만 선언하고 나중에 할당
    - 이런 경우 타입 검사기가 data를 정확히 추론한다.

In [ ]:
data: dict[str, int]
...
data = {"a": 1}

- 클래스-level constant

In [ ]:
class Config:
    DEFAULT_PORT: int = 8080

- `Final` (변경 불가 변수) 자주 사용

In [ ]:
from typing import Final

API_VERSION: Final[str] = "v1"